# Teresio - MinHashing

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

subset = "horoscope_full"
df = pd.read_csv(f'..\\data\\{subset}.csv')
df.head()

,ID,sign,category,date,horoscope
0,1,aries,general,20200617,"There's a great day ahead of you, Aries. You'l..."
1,2,aries,general,20200618,People will understand and appreciate your des...
2,3,aries,general,20200619,You are very interested in technological break...
3,4,aries,general,20200620,Stress from overwork could have you feeling we...
4,5,aries,general,20200621,This is a good day to stand up for yourself an...


In [3]:
df_general = df[df["category"] == "general"]

horoscope_general_dict = {
    row["ID"]: row["horoscope"]
    for _, row in df_general.iterrows()
}

In [3]:
import re

def normalize_text(text):
    text = text.lower()
    
    # 1. Remove punctuation and replace with a space
    text = re.sub(r"[^\w\s]", " ", text) # Removes everything that is not a word character or whitespace
    
    # 2. Remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    # 3. Return the list of words (tokens)
    return text.split(' ')


def shingle(q, text):
    newtext = normalize_text(text)
    shingles = []
    # The index must go from 0 to the last shingle index. 
    # The last shingle index is N (number of words) - q. 
    # We add +1 so it's not excluded by the Python for loop's range.
    for i in range(len(newtext) - q + 1): 
        # The join method combines elements of a list into a single string, 
        # separated by a space character (" "). In this case, 
        # the list elements go from index i to index i+q.
        shingles.append(" ".join(newtext[i:i + q])) 
    
    # Remove repetitions
    unique = set(shingles) 
    # Convert back to a list
    list_shingles = list(unique) 
    
    return list_shingles

In [4]:
import sys
import os
import mmh3


#################### Utilities ######################
#hashes a list of strings
def listhash(l,seed):
	val = 0
	for e in l:
		val = val ^ mmh3.hash(e, seed)
	return val 

In [5]:
def signature(docs: dict, seedlist: list, q: int):
    
    
    doc_ids = list(docs.keys()) #estraggo id del dizionario dall dict, ora ho una lista con ID del dizionario
    k = len(seedlist) # k è il numero di funzioni hash
    
    
    shingle_to_docs = {} #preparo un dict vuoto
    
    # Ciclo iniziale per popolare l'indice e preparare gli shingles
    for doc_id, text in docs.items(): #associo in doc_id ID dei documenti e in text il testo dei documenti
        
        shingles_list = shingle(q, text) #preparo la lista di shingles, questo per un documento 
        for shingle_item in shingles_list:
          
            shingle_to_docs.setdefault(shingle_item, set()).add(doc_id)

        # il doppio ciclo mi serve per scorrere tutti i documenti e in ogni documento ogni shingle
        # ho la mia lista vuota a cui applico la funzione setdefault questa funzione prende la key (uno shingle) 
        # e la cerca all'interno del dizionario se c'è aggiunge un altro documento ID nella raccolta set dei value, 
        # se non c'è crea questa chiave e aggiunge L'ID del documento immediatamente all'interno di un oggetto set  
        # quello che ho alla fine è un grande dictionary con tutti gli shingles come key e i documenti dove sono presenti come value
         

    doc_signatures = {              #definisco il dizionario delle firme con id come key e una lista lunga quanto le funzioni hash con all'interno solo inf, quindi k inf
        doc_id: [float("inf")] * k  #[float("inf")] * k  quando moltiplichi una lista contenente un singolo elemento per un numero intero, il risultato 
                                    #è una nuova lista in cui quell'elemento è ripetuto k volte.
        for doc_id in doc_ids
    } # quindi ora ho un dict inizializzato con le key dei documenti e come value una lista lunga quanto il numero di f.ash di infinito
    
    # a questo punto ho un dict che ha come key gli shingles e come value la lista dove lo shingle si trova shingle_to_docs 
    # inoltre ho un dict doc_signatures che ha come key gli id dei documenti e come value invece una lista di infiniti tanti quante le funzioni hash 
    
    for shingle_item in shingle_to_docs.keys(): # prendi lo shingle (key) dalla lista di shingle (ciclato su tutti gli shingle)
        
        hash_values = [listhash(shingle_item, seed) for seed in seedlist] # prendo lo shingle e gli applico tutte l hash function e le salvo in una lista
        
        for doc_id in shingle_to_docs[shingle_item]: #scorri tutti gli ID dove trovi questo shingle. shingle_to_docs[shingle_item] è il value ovvero la lista ID dove puoi
                                                     #trovare lo shingle e scorri tutti gli ID
            current_signature = doc_signatures[doc_id]# la firma corrente è la prima volta tutti infiniti e poi si aggiorna con tutti i minimi
            for i in range(k):
                current_signature[i] = min(current_signature[i], hash_values[i])
                
    #prendo lo shingle e gli applico tutte le funzioni hash e le salvo in una lista che avrà lunghezza k
    #prendo la lista dei documenti dove questo shingle è presente
    # per ogni documento in cui è presente cerco la sua firma attuale
    # controllo funzione per funzione se l'ash che ho adesso è minore nel caso sostiuisco
    # quindi per ogni shingles, per ogni documento in cui è contenuto e per ogni funzione ash.            
    return doc_signatures

In [6]:
import numpy as np
def jaccard (doc_id_1: str, doc_id_2: str, doc_signatures: dict):
    sign1 = np.array(doc_signatures[doc_id_1])
    sign2 = np.array(doc_signatures[doc_id_2])
    matches = 0
    k = len(sign1)
    
    for i in range(k):
        if sign1[i] == sign2[i]:
            matches += 1
            
    similarity = matches / k
    return similarity

In [7]:
def similar(doc_signatures, threshold):
    list_keys = list(doc_signatures.keys()) # I save all the keys of the documents in a list
    similar_items = {} # empty dict to save the two ids and the similarity value
    
    # double for cycle with i less than j so as not to repeat the combinations
    for i in range (len(list_keys)-1): 
        for j in range (i + 1, len(list_keys)):
            similarity_score = jaccard(list_keys[i], list_keys[j], doc_signatures)
            if similarity_score >= threshold:
                # as key I put the two IDs of the documents, as value the similarity value
                similar_items[(list_keys[i], list_keys[j])] = similarity_score 
                
    return similar_items

In [8]:
def lsh(signatures_dict, b, jaccard_threshold=0.5, seed=42):
    lsh_dict = {} # new dict that has as key the IDs and as value the hashes of the blocks
    for key, values in signatures_dict.items(): # for each item with its key value ID: signature
        blocks = np.split(np.array(values), b) # split the signature into blocks
        blocks_hash_values = [] # empty list for the new signature
        for aBlock in blocks: # for each block among the blocks
            band_bytes = aBlock.tobytes()
            # hash for each block until a list of hashes is created
            blocks_hash_values.append(mmh3.hash(band_bytes, seed)) 
        # in the dict I put the ID in the key and the new list of hashed blocks in the value
        lsh_dict[key] = blocks_hash_values 
        
    list_keys = list(lsh_dict.keys()) # I save the list of keys
    similar_items = {} # new dict
    
    for i in range (len(list_keys)-1):
        for j in range (i+1, len(list_keys)):
            # how many in common in the new list?
            common_values = np.intersect1d(lsh_dict[list_keys[i]], lsh_dict[list_keys[j]]) 
            
            # if at least one then they are candidates and I calculate them with jaccard
            if len(common_values) > 0: 
                # we found a candidate
                similarity_score = jaccard(list_keys[i], list_keys[j], signatures_dict)
                
                # if they exceed the threshold
                if similarity_score >= jaccard_threshold: 
                    # the key of similar items are the name of the two documents and the value is the similarity
                    similar_items[(list_keys[i], list_keys[j])] = similarity_score 
                    
    return similar_items

GENERAL CATEGORY


In [ ]:
# Parameters

q = 3     # size of the shingle (in words)                   
                            
r = 3      # rows per band                   
                           
k = 120                     # number of hash functions
                            
b = k // r                  # number of bands 
                            
threshold = 0.30            # minimum similarity threshold 
                            
seedlist = list(range(k))   # list of seeds for the hash functions 
                            
seed_lsh = 42               # seed for hashing the blocks 
                            
# Compute MinHash signatures on all documents
full_signatures = signature(horoscope_general_dict, seedlist, q)

# Find similar items using the LSH function

similar_items_lsh = lsh(full_signatures, b, jaccard_threshold=threshold, seed=seed_lsh)


# ---------------- Stampa risultati ----------------
# Print results

for (id1, id2), sim in similar_items_lsh.items():
    print(f"ID1: {id1}, ID2: {id2}, Similarità: {sim:.2f}")
    print(f"Text1: {horoscope_general_dict[id1]}")
    print(f"Text2: {horoscope_general_dict[id2]}")
    print("------")

ID1: 25, ID2: 356, Similarità: 0.53
Text1: It may be difficult to stay grounded today with all the information flying around and the emotions roiling in your heart. Try not to take things too seriously, Aries. This is the key to maintaining a level head throughout the day. Do things with passion and take care of any detective work that needs to be done. There are important facts coming to you from unexpected sources.
Text2: It may be difficult to stay grounded today with all the information flying around and all the emotion roiling in your heart. Try not to take things too seriously, Aries. This is the key to maintaining a level head today. Approach the day with passion and take care of any investigative work that needs doing. There are important facts coming from unexpected sources.
------
ID1: 54, ID2: 247, Similarità: 0.33
Text1: Changes taking place in your home could cause some temporary frustration tense nerves on the part of family members, Aries. Perhaps you're moving or refurn

In [21]:
series = pd.Series(similar_items_lsh)

# 2. Converti l'indice in un DataFrame (ID1 e ID2)
# .index.tolist() estrae le tuple [(25, 356), (54, 247), ...]
df = pd.DataFrame(series.index.tolist(), columns=['ID1', 'ID2'])

# 3. Aggiungi i valori di similarità come una nuova colonna
# I valori della Series (series.values) sono i valori di similarità.
df['Similarity'] = series.values

print("## final table")
print(df)
df.to_csv('similarity_general_lsh.csv', index=False)

## final table
      ID1    ID2  Similarity
0      25    356    0.533333
1      54    247    0.325000
2      55    248    0.858333
3      63    255    0.500000
4      64    256    0.408333
..    ...    ...         ...
78  20184  20377    0.566667
79  20185  20378    0.483333
80  20193  20385    0.358333
81  20250  20297    0.558333
82  20444  20494    0.583333

[83 rows x 3 columns]


In [16]:
print("dataset lenght", len(horoscope_general_dict))
print("\n")

def counter_pair_sim(similar_items_lsh, sim):
    pair_sim = {paair: similar for paair, similar in similar_items_lsh.items() if similar>= sim } 
    lunghezza_dict = len(pair_sim)
    return lunghezza_dict

thresholds = [1, 0.9, 0.75, 0.5, 0.3]

for t in thresholds:
    count = counter_pair_sim(similar_items_lsh, t)
    print("Number of couples with similarity ≥", t, ": ",  count, "\n")

dataset lenght 4391


Number of couples with similarity ≥ 1 :  1 

Number of couples with similarity ≥ 0.9 :  1 

Number of couples with similarity ≥ 0.75 :  6 

Number of couples with similarity ≥ 0.5 :  39 

Number of couples with similarity ≥ 0.3 :  83 



In [19]:
def oroscope_least1(similar_items_lsh, sim):
    pair_sim = {paair: similar for paair, similar in similar_items_lsh.items() if similar>= sim }
    horoscope_least_1_sim = set()
    for pair in pair_sim.keys():
        horoscope_least_1_sim.update(pair)
    return len(horoscope_least_1_sim)



for t in thresholds:
    print("horoscope rycicled at least once with similarity >=", t, ": ",  oroscope_least1(similar_items_lsh, t))
    print("lower limit percentage of recycled dataset >=", t, ": ",  (oroscope_least1(similar_items_lsh, t)/2)/len(horoscope_general_dict)*100)
    print("\n")

horoscope rycicled at least once with similarity >= 1 :  2
lower limit percentage of recycled dataset >= 1 :  0.02277385561375541


horoscope rycicled at least once with similarity >= 0.9 :  2
lower limit percentage of recycled dataset >= 0.9 :  0.02277385561375541


horoscope rycicled at least once with similarity >= 0.75 :  12
lower limit percentage of recycled dataset >= 0.75 :  0.13664313368253245


horoscope rycicled at least once with similarity >= 0.5 :  78
lower limit percentage of recycled dataset >= 0.5 :  0.8881803689364609


horoscope rycicled at least once with similarity >= 0.3 :  166
lower limit percentage of recycled dataset >= 0.3 :  1.890230015941699




LOVE CATEGORY

In [12]:
df_general = df[df["category"] == "love"]

horoscope_love_dict = {
    row["ID"]: row["horoscope"]
    for _, row in df_general.iterrows()
}



In [14]:
# Parameters

q = 3     # size of the shingle (in words)                   
                            
r = 3      # rows per band                   
                           
k = 120                     # number of hash functions
                            
b = k // r                  # number of bands 
                            
threshold = 0.30            # minimum similarity threshold 
                            
seedlist = list(range(k))   # list of seeds for the hash functions 
                            
seed_lsh = 42               # seed for hashing the blocks 
                            
# Compute MinHash signatures on all documents
full_signatures = signature(horoscope_love_dict, seedlist, q)

# Find similar items using the LSH function

love_similar_items_lsh = lsh(full_signatures, b, jaccard_threshold=threshold, seed=seed_lsh)


# ---------------- Stampa risultati ----------------
# Print results

for (id1, id2), sim in love_similar_items_lsh.items():
    print(f"ID1: {id1}, ID2: {id2}, Similarità: {sim:.2f}")
    print(f"Text1: {horoscope_love_dict[id1]}")
    print(f"Text2: {horoscope_love_dict[id2]}")
    print("------")

ID1: 366, ID2: 20861, Similarità: 1.00
Text1: Love is deeply private and very passionate today, with the current planetary configuration. Even if everyone knows of your relationship with a certain person, they certainly do not realize how deeply in love you actually are, and perhaps you do not either until an event that occurs today makes you realize exactly what is going on. You may choose to keep this a secret a while longer.
Text2: Love is deeply private and very passionate today, with the current planetary configuration. Even if everyone knows of your relationship with a certain person, they certainly do not realize how deeply in love you actually are, and perhaps you do not either until an event that occurs today makes you realize exactly what is going on. You may choose to keep this a secret a while longer.
------
ID1: 382, ID2: 8034, Similarità: 1.00
Text1: Perhaps it is a lack of vision by you or your partner (current or prospective) that creates an emotional problem. The curre

In [15]:
print("dataset lenght", len(horoscope_love_dict))
print("\n")

def counter_pair_sim(similar_items_lsh, sim):
    pair_sim = {paair: similar for paair, similar in love_similar_items_lsh.items() if similar>= sim } 
    lunghezza_dict = len(pair_sim)
    return lunghezza_dict

thresholds = [1, 0.9, 0.75, 0.5, 0.3]

for t in thresholds:
    count = counter_pair_sim(love_similar_items_lsh, t)
    print("Number of couples with similarity ≥", t, ": ",  count, "\n")

dataset lenght 4392


Number of couples with similarity ≥ 1 :  329 

Number of couples with similarity ≥ 0.9 :  374 

Number of couples with similarity ≥ 0.75 :  463 

Number of couples with similarity ≥ 0.5 :  540 

Number of couples with similarity ≥ 0.3 :  546 



In [19]:
def oroscope_least1(similar_items_lsh, sim):
    pair_sim = {paair: similar for paair, similar in similar_items_lsh.items() if similar>= sim }
    horoscope_least_1_sim = set()
    for pair in pair_sim.keys():
        horoscope_least_1_sim.update(pair)
    return len(horoscope_least_1_sim)



for t in thresholds:
    print("horoscope rycicled at least once with similarity >=", t, ": ",  oroscope_least1(love_similar_items_lsh, t))
    print("lower limit percentage of recycled dataset >=", t, ": ",  (oroscope_least1(love_similar_items_lsh, t)/2)/len(horoscope_love_dict)*100)
    print("\n")

horoscope rycicled at least once with similarity >= 1 :  658
lower limit percentage of recycled dataset >= 1 :  7.490892531876138


horoscope rycicled at least once with similarity >= 0.9 :  748
lower limit percentage of recycled dataset >= 0.9 :  8.515482695810565


horoscope rycicled at least once with similarity >= 0.75 :  926
lower limit percentage of recycled dataset >= 0.75 :  10.541894353369763


horoscope rycicled at least once with similarity >= 0.5 :  1080
lower limit percentage of recycled dataset >= 0.5 :  12.295081967213115


horoscope rycicled at least once with similarity >= 0.3 :  1092
lower limit percentage of recycled dataset >= 0.3 :  12.431693989071038




In [20]:
series = pd.Series(love_similar_items_lsh)

# 2. Converti l'indice in un DataFrame (ID1 e ID2)
# .index.tolist() estrae le tuple [(25, 356), (54, 247), ...]
df = pd.DataFrame(series.index.tolist(), columns=['ID1', 'ID2'])

# 3. Aggiungi i valori di similarità come una nuova colonna
# I valori della Series (series.values) sono i valori di similarità.
df['Similarity'] = series.values

print("## final table")
print(df)
df.to_csv('similarity_love_lsh.csv', index=False)

## final table
       ID1    ID2  Similarity
0      366  20861    1.000000
1      382   8034    1.000000
2      383   8035    1.000000
3      384   8036    0.783333
4      385   8037    1.000000
..     ...    ...         ...
541  13206  20858    0.950000
542  13541  15006    1.000000
543  15371  16836    1.000000
544  17201  18666    1.000000
545  19031  20496    1.000000

[546 rows x 3 columns]
